# SurviveCity v2 — Kaggle T4 (fixed reward function)

This notebook fixes the no-signal stalemate observed in run `aea94d692002`
(loss=0, reward_std=0, grad_norm=0 for every step). Root cause: the previous
`reward_fn` returned the LAST step's clipped value after a 99-step random
rollout — terminal death always clipped to 0.01, so every completion in the
GRPO group got the same floor and the gradient was zero.

**What's different here:**

1. `reward_fn` returns `5*step1_raw + cumulative_raw_over_30_steps + 0.10*format_bonus`
   (no clipping in the reward fn — TRL normalises within the group anyway).
2. Rollout shortened from 99 → 30 steps so the model's single action has
   leverage instead of being drowned out by random play.
3. Per-call diagnostics print parse-success rate, action-type histogram,
   and reward mean/std/min/max so you can SEE learning happen in the log.
4. Pinned dep stack carried over from the DGX saga (torchao removed,
   llm_blender stubbed, `TRANSFORMERS_CACHE` patched).
5. Smoke test before training: scores 5 dummy completions and asserts
   reward_std > 0.01 — fails fast if anything broke the fix.

Expected behaviour:
- After step 1, `train/reward_std > 0` and `train/loss > 0`.
- After step 5, action distribution shifts toward a few useful types.
- After step 10, mean reward trends up.

If `reward_std` is still 0 at step 1, kill the run — something is wrong.


## 0. Fix CUDA_VISIBLE_DEVICES BEFORE any imports

Kaggle gives you 2x T4 by default. `bitsandbytes` + DataParallel breaks
CUBLAS in this combination. Bind to one device before torch is even
imported (after-the-fact `os.environ` assignment doesn't take effect
because torch caches the device count on first import).


In [ ]:
import os, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("env set: CUDA_VISIBLE_DEVICES=", os.environ["CUDA_VISIBLE_DEVICES"])
print("python:", sys.version.split()[0])


## 1. Pull HF + GitHub tokens from Kaggle secrets (or env)

Tokens are NEVER written to disk. If Kaggle Secrets isn't set, the cell
falls back to plain `os.environ` so you can do `%env HF_TOKEN=hf_xxx`
in a previous cell on Colab/local without changing this notebook.


In [ ]:
HF_TOKEN = None
GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    try: HF_TOKEN = _sec.get_secret("HF_TOKEN")
    except Exception: pass
    try: GITHUB_TOKEN = _sec.get_secret("GITHUB_TOKEN")
    except Exception: pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
GITHUB_TOKEN = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print(f"HF_TOKEN set ({len(HF_TOKEN)} chars, prefix={HF_TOKEN[:6]}...)")
else:
    print("WARNING: no HF_TOKEN — push_to_hub will fail. Add it to Kaggle Secrets as 'HF_TOKEN'.")
print(f"GITHUB_TOKEN: {'set' if GITHUB_TOKEN else 'absent (fine for public repo clone)'}")


## 2. Pin torch cu121 FIRST

Reason: `accelerate==1.1.1` (which we install in the next cell) silently
replaces the Kaggle-shipped torch with the PyPI CPU wheel during dependency
resolution. Pin torch first with `--upgrade-strategy=only-if-needed` so
later installs don't downgrade it. If you skip this, you'll spend 30 min
training before noticing 'CUDA not available' in the logs.


In [ ]:
import subprocess
def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1500:] if r.stdout else "")
    if r.stderr: print("STDERR:", r.stderr[-800:])
    if check and r.returncode != 0: raise RuntimeError(f"pip failed (rc={r.returncode})")
    return r

if "torch" in sys.modules:
    raise RuntimeError(
        "torch is already imported in this kernel. Restart it: 'Run' menu -> "
        "'Factory reset' (or 'Restart & Clear Output'). Re-import below."
    )

pip("install", "-q",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "--upgrade-strategy=only-if-needed",
    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121", "torchaudio==2.5.1+cu121")


## 3. Install pinned dep stack

Versions chosen from the DGX saga (these are the ones that don't break
each other):
  - transformers 4.46.3   (trl 0.15+ needs >=4.46.0)
  - peft 0.13.2           (last version before the torchao>=0.16 demand)
  - trl 0.15.2            (first stable version with GRPOTrainer exposed)
  - bitsandbytes 0.43.3   (fp4/nf4 stable on T4)
  - accelerate 1.1.1      (pinned so the resolver doesn't pull a newer one)
  - datasets 3.1.0        (pinned to match transformers 4.46)
  - mergekit              (chain-imported by trl 0.15+)


In [ ]:
pip("install", "-q",
    "--upgrade-strategy=only-if-needed",
    "transformers==4.46.3",
    "peft==0.13.2",
    "trl==0.15.2",
    "bitsandbytes==0.43.3",
    "accelerate==1.1.1",
    "datasets==3.1.0",
    "mergekit",
    "huggingface_hub>=0.26",
    "tensorboard",
    "tbparse",
    "psutil",
    "pydantic")


## 4. Post-install fixes for known breakage

- `torchao`: peft 0.13 demands >=0.16, but torch 2.5 only ships <=0.7.
  Solution: uninstall it entirely. peft's check returns False cleanly when
  it's absent.
- `llm_blender`: a transitive dep that imports `transformers.TRANSFORMERS_CACHE`
  (which 4.46 removed). Stub it as a 6-line fake module so the import
  succeeds and trl's optional-import returns False.
- transformers `TRANSFORMERS_CACHE`: a few internal sites still reference
  the removed name. Patch the import to alias it to `HF_HUB_CACHE`.


In [ ]:
# 1) torchao — kill it
pip("uninstall", "-y", "torchao", check=False)

# 2) llm_blender — replace with a stub so the optional import returns False
import site, pathlib
sp = pathlib.Path(site.getusersitepackages() if hasattr(site,"getusersitepackages") else site.getsitepackages()[0])
# Pick the first writable site-packages dir
for cand in [pathlib.Path(p) for p in (site.getsitepackages() if hasattr(site,"getsitepackages") else []) + ([site.getusersitepackages()] if hasattr(site,"getusersitepackages") else [])]:
    try:
        cand.mkdir(parents=True, exist_ok=True)
        (cand / ".touch_test").write_text("x"); (cand / ".touch_test").unlink()
        sp = cand; break
    except Exception:
        continue
print("site-packages target:", sp)
stub = sp / "llm_blender"
stub.mkdir(parents=True, exist_ok=True)
(stub / "__init__.py").write_text(
"\"\"\"Stub: real llm_blender breaks on transformers 4.46 (TRANSFORMERS_CACHE removal).\"\"\"\n"
"class Blender:\n"
"    def __init__(self,*a,**kw): raise ImportError('llm_blender stubbed in this env')\n"
)
print("stub written:", stub / "__init__.py")

# 3) transformers TRANSFORMERS_CACHE alias
import transformers.utils.hub as _hub
if not hasattr(_hub, "TRANSFORMERS_CACHE"):
    _hub.TRANSFORMERS_CACHE = getattr(_hub, "HF_HUB_CACHE", None) or os.path.expanduser("~/.cache/huggingface/hub")
    print(f"patched TRANSFORMERS_CACHE -> {_hub.TRANSFORMERS_CACHE}")

# 4) Verify versions actually took. Kaggle's pre-installed transformers can
#    survive our `pip install transformers==4.46.3` (the system dist-packages
#    isn't always cleanly replaceable from a notebook). If the loaded
#    transformers is newer than 4.46.3, it will demand a newer bitsandbytes
#    than our pin (e.g. >=0.46.1) — and the model load in main() will crash
#    AFTER we've spent 30 min downloading weights. Catch + auto-correct now.
import importlib, importlib.metadata as _meta
import transformers as _tf
print(f"transformers loaded from: {_tf.__file__}")
print(f"transformers version:     {_tf.__version__}")
print(f"bitsandbytes version:     {_meta.version('bitsandbytes')}")
try:
    from transformers.utils.import_utils import is_bitsandbytes_available
    if not is_bitsandbytes_available():
        print("bnb compatibility check FAILED — upgrading bitsandbytes to whatever satisfies this transformers...")
        pip("install", "-q", "-U", "bitsandbytes")
        import bitsandbytes as _bnb
        importlib.reload(_bnb)
        print(f"  bitsandbytes upgraded to: {_bnb.__version__}")
        # Re-check
        from transformers.utils.import_utils import is_bitsandbytes_available as _ib2
        assert _ib2(), "bnb still not satisfying the transformers minimum after upgrade — restart kernel."
        print("  bnb compatibility OK now.")
    else:
        print("bnb compatibility OK (no upgrade needed).")
except ImportError as _e:
    print(f"  could not import is_bitsandbytes_available: {_e}")


## 5. Verify torch + GPU

If this prints `cuda: False` or device count != 1, stop here and restart
the kernel — something replaced torch with a CPU build.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), " device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0), "cc=", torch.cuda.get_device_capability(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM free={free/1e9:.1f}GB total={total/1e9:.1f}GB")
assert torch.cuda.is_available(), "CUDA not available — restart kernel and re-run cells from the top."


## 6. Clone repo + cd to v2

Uses `GIT_TERMINAL_PROMPT=0` and `GIT_ASKPASS=/bin/echo` so a missing token
fails fast (instead of hanging waiting for a username). If the repo is
public this works without any token. If it's private, set GITHUB_TOKEN in
Kaggle Secrets — the cell embeds it via `http.extraheader` rather than
URL-baking, so it doesn't end up in `.git/config`.


In [ ]:
os.environ["GIT_TERMINAL_PROMPT"] = "0"
os.environ["GIT_ASKPASS"] = "/bin/echo"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp"
REPO_DIR = os.path.join(WORK, "zombiee")
if os.path.isdir(REPO_DIR):
    print("repo already cloned at", REPO_DIR, "— pulling latest")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    cmd = ["git", "clone", "--depth=1", "https://github.com/SirjanSingh/zombiee.git", REPO_DIR]
    if GITHUB_TOKEN:
        # http.extraheader avoids baking the token into the cloned repo's config
        auth = f"Authorization: Basic {__import__('base64').b64encode(f'x-access-token:{GITHUB_TOKEN}'.encode()).decode()}"
        cmd = ["git", "-c", f"http.extraheader={auth}"] + cmd[1:]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    assert r.returncode == 0, "git clone failed (set GITHUB_TOKEN if repo is private)"
V2 = os.path.join(REPO_DIR, "v2")
assert os.path.isdir(V2), f"v2/ not found in repo at {V2}"
os.chdir(V2)
sys.path.insert(0, V2)
print("CWD:", os.getcwd())


## 7. Verify (or apply) the reward-fix patch

If master already contains `REWARD_FN_VERSION = "v2-cumulative-2026-04-26"`
we're done. Otherwise we patch `training/train.py` in place — this lets the
notebook work even if the fix hasn't been pushed to GitHub yet.


In [ ]:
# Inline patch: replace create_reward_fn in training/train.py if missing the v2 marker
import re as _re, pathlib as _pl
_train_py = _pl.Path("training/train.py")
_src = _train_py.read_text()
if "v2-cumulative-2026-04-26" in _src:
    print("OK: train.py already has the v2-cumulative reward fix.")
else:
    print("PATCHING: train.py is the old version, replacing create_reward_fn ...")
    _new_fn = """# Marker used by the kaggle notebook to detect whether the file already has
# the v2-reward-fix applied; do not remove.
REWARD_FN_VERSION = "v2-cumulative-2026-04-26"


def create_reward_fn():
    # GRPO reward function — v2-cumulative (inline-patched by notebook).
    from collections import Counter
    import statistics
    from survivecity_v2_env.env import SurviveCityV2Env
    from training.inference import (
        parse_action, RANDOM_NON_VOTE_ACTIONS,
    )
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = lambda x, **kw: x

    state = {"calls": 0, "errors": 0}
    ROLLOUT_LIMIT = 30
    FORMAT_BONUS = 0.10
    STEP1_WEIGHT = 5.0

    def reward_fn(prompts, completions, **kwargs):
        state["calls"] += 1
        rewards = []
        action_types = []
        rollout_lens = []
        parse_ok_count = 0
        n = len(prompts)
        iterator = tqdm(list(zip(prompts, completions)), total=n,
                        desc=f"reward_fn #{state['calls']}", leave=False)
        for prompt, completion in iterator:
            try:
                seed_match = re.search(r"\\[SEED:(\\d+)\\]", prompt)
                ep_seed = int(seed_match.group(1)) if seed_match else (
                    abs(hash(prompt)) % 1_000_000
                )
                env = SurviveCityV2Env()
                obs = env.reset(seed=ep_seed)
                parsed = parse_action(completion, agent_id=0)
                parse_ok = parsed is not None
                if parse_ok:
                    parse_ok_count += 1
                    action = parsed
                    action_types.append(parsed.get("action_type", "?"))
                else:
                    action = {"agent_id": 0, "action_type": "wait"}
                    action_types.append("PARSE_FAIL")
                obs = env.step(action)
                step1_raw = obs.get("metadata", {}).get("raw_reward", 0.0)
                rollout_rng = random.Random(ep_seed + 7)
                steps = 0
                while not obs.get("done", False) and steps < ROLLOUT_LIMIT:
                    aid = obs.get("metadata", {}).get("current_agent_id", 0)
                    sc = obs.get("step_count", 0)
                    if sc in (30, 60, 90):
                        rand_act = {"agent_id": aid, "action_type": "vote_lockout",
                                    "vote_target": rollout_rng.choice([0, 1, 2, 3, 4])}
                    else:
                        rand_act = {"agent_id": aid,
                                    "action_type": rollout_rng.choice(RANDOM_NON_VOTE_ACTIONS)}
                    obs = env.step(rand_act)
                    steps += 1
                cum0 = obs.get("metadata", {}).get("cumulative_rewards", {}).get(0, 0.0)
                composite = (STEP1_WEIGHT * step1_raw + cum0 +
                             (FORMAT_BONUS if parse_ok else 0.0))
                rewards.append(float(composite))
                rollout_lens.append(steps)
            except Exception as e:
                state["errors"] += 1
                if state["errors"] <= 5 or state["errors"] % 50 == 0:
                    logger.warning(
                        f"reward_fn error #{state['errors']} ({type(e).__name__}): {e}"
                    )
                rewards.append(0.0)
                action_types.append("ERROR")
                rollout_lens.append(0)
        try:
            r_mean = statistics.mean(rewards)
            r_std = statistics.stdev(rewards) if len(rewards) > 1 else 0.0
            ro_mean = statistics.mean(rollout_lens) if rollout_lens else 0.0
        except statistics.StatisticsError:
            r_mean = r_std = ro_mean = 0.0
        action_dist = Counter(action_types).most_common(6)
        action_dist_str = ", ".join(f"{a}={c}" for a, c in action_dist)
        logger.info(
            f"reward_fn #{state['calls']}: n={n} "
            f"parse_ok={parse_ok_count}/{n} "
            f"r[mean={r_mean:+.3f} std={r_std:.3f} min={min(rewards):+.3f} max={max(rewards):+.3f}] "
            f"avg_rollout={ro_mean:.0f} "
            f"actions[{action_dist_str}] "
            f"errs={state['errors']}"
        )
        return rewards
    return reward_fn
"""
    # Use a lambda replacement so backslashes inside _new_fn (e.g. regex
    # patterns like r"\[SEED:(\d+)\]") are NOT interpreted as backref
    # templates by re.sub.
    _patched = _re.sub(
        r"def create_reward_fn\(\).*?(?=\ndef _resolve_resume\b)",
        lambda _m: _new_fn + "\n\n",
        _src,
        count=1, flags=_re.DOTALL,
    )
    if _patched == _src:
        raise RuntimeError(
            "Could not locate create_reward_fn in training/train.py to patch. "
            "Check the file manually."
        )
    _train_py.write_text(_patched)
    print(f"  wrote patched train.py ({len(_patched)} chars)")
    # Confirm marker now present
    assert "v2-cumulative-2026-04-26" in _train_py.read_text(), "patch did not stick"
    print("OK: patch applied + verified.")


## 8. Smoke test — non-zero reward variance

Score 5 dummy completions (mix of clean JSON, prose with embedded action
words, and pure prose with no valid action). The new reward fn must
produce reward_std > 0.01 on this batch — that's the GRPO gradient floor.
If this fails, training will silently no-op for 4 hours; better to find
out now.


In [ ]:
import random as _rnd, re as _re_, statistics as _stat
from survivecity_v2_env.env import SurviveCityV2Env
from training.inference import parse_action, RANDOM_NON_VOTE_ACTIONS

samples = [
    '{"action_type": "eat"}',
    '{"action_type": "move_up"}',
    '{"action_type": "wait"}',
    "I'll move_left to safety since the zombie is at (5,5).",
    "I think we should think about strategy more.",  # no valid action -> parse_fail
]
env0 = SurviveCityV2Env(); obs0 = env0.reset(seed=42)
prompt = f"[SEED:42]\n{obs0.get('description','')[:200]}"

ROLLOUT, BONUS, W1 = 30, 0.10, 5.0
def score(c):
    s = int(_re_.search(r"\[SEED:(\d+)\]", prompt).group(1))
    e = SurviveCityV2Env(); o = e.reset(seed=s)
    p = parse_action(c, agent_id=0); ok = p is not None
    a = p if ok else {"agent_id": 0, "action_type": "wait"}
    o = e.step(a); s1 = o.get("metadata",{}).get("raw_reward",0.0)
    rng = _rnd.Random(s + 7); k = 0
    while not o.get("done",False) and k < ROLLOUT:
        aid = o.get("metadata",{}).get("current_agent_id",0)
        sc = o.get("step_count",0)
        if sc in (30,60,90):
            ra = {"agent_id": aid, "action_type": "vote_lockout", "vote_target": rng.choice([0,1,2,3,4])}
        else:
            ra = {"agent_id": aid, "action_type": rng.choice(RANDOM_NON_VOTE_ACTIONS)}
        o = e.step(ra); k += 1
    cum = o.get("metadata",{}).get("cumulative_rewards",{}).get(0,0.0)
    return ok, (p.get("action_type") if ok else "PARSE_FAIL"), s1, cum, W1*s1+cum+(BONUS if ok else 0)

print(f"  {'completion':<55s}  parse  action       step1     cum0     composite")
rs = []
for c in samples:
    ok, a, s1, cum, comp = score(c)
    print(f"  {c[:50]:<53s}  {ok!s:<5s}  {a:<12s} {s1:+.3f}   {cum:+.3f}   {comp:+.3f}")
    rs.append(comp)
print(f"\n  group mean={_stat.mean(rs):+.3f}  std={_stat.stdev(rs):.3f}  min={min(rs):+.3f}  max={max(rs):+.3f}")
assert _stat.stdev(rs) > 0.01, "reward_std too low — GRPO will not get a gradient. Investigate before training."
print("\n  PASS — reward variance is above GRPO gradient floor; safe to train.")


## 9. Pre-create the Hub repo (idempotent)

Avoids the race where `hub_strategy="every_save"` tries to push to a repo
that doesn't exist yet on the very first save.


In [ ]:
HUB_REPO = os.environ.get("HUB_REPO", "noanya/zombiee-v2")
print("Hub target:", HUB_REPO)
from huggingface_hub import HfApi, login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi()
    try:
        api.create_repo(repo_id=HUB_REPO, repo_type="model", exist_ok=True, private=True)
        print(f"  hub repo ready: https://huggingface.co/{HUB_REPO}")
    except Exception as e:
        print(f"  WARN: create_repo failed ({type(e).__name__}: {e}). Push may fail.")
else:
    print("  no HF_TOKEN — skipping repo precreation; push_to_hub will be disabled.")


## 10. Train

Hyperparameters tuned for visible learning on a T4:
  - `--max-steps 20`        more datapoints than the broken run had
  - `--save-steps 2`        save every other step (10 checkpoints on Hub)
  - `--num-generations 8`   slightly smaller group than 12 → faster steps
  - `--temperature 1.1`     widen completion distribution → wider rewards
  - `--max-completion-length 256`  shorter generations → faster steps
  - `--max-prompt-length 1024`     trim prompt overhead
  - `--num-scenarios 80`           smaller dataset is fine for 20 steps
  - `--grad-accum-steps 2`         effective batch = 16 evals/step

Per-step time on T4 should drop from ~16 min to ~8-10 min. Total ≈ 3 hours
for 20 steps + push overhead.

The trainer will print one `[metrics] step N: {...}` line per step AND one
`reward_fn #K: ... r[mean=... std=...] actions[...]` line per call. Watch
the `std` and `actions[]` columns — if std stays at 0 or the action
distribution is dominated by `PARSE_FAIL` past step 3, kill the run.


In [ ]:
import importlib, training.train as _train_mod
importlib.reload(_train_mod)  # pick up the patched create_reward_fn

ARGS = [
    "--model-name", "Qwen/Qwen2.5-3B-Instruct",
    "--max-steps", "20",
    "--save-steps", "2",
    "--save-total-limit", "10",
    "--num-generations", "8",
    "--per-device-batch-size", "1",
    "--grad-accum-steps", "2",
    "--max-prompt-length", "1024",
    "--max-completion-length", "256",
    "--temperature", "1.1",
    "--lora-r", "16",
    "--lora-alpha", "32",
    "--lr", "1e-5",
    "--num-scenarios", "80",
    "--output-dir", os.path.join(WORK, "ckpts_v2_fixed"),
    "--push-to-hub",
    "--hub-model-id", HUB_REPO,
]
if HF_TOKEN: ARGS += []  # token is in env, GRPOTrainer reads it
sys.argv = ["train.py"] + ARGS
print("Launching training with args:")
for a in ARGS: print("  ", a)
print()

_train_mod.main()


## 11. Post-training summary

Parse the just-written TB events and print every step's metrics + the
per-step reward variance. If `reward_std` stays >0 across all 20 steps,
training was healthy. If it collapses to 0 mid-run, the LoRA hit a degen
solution (e.g. all completions identical) — flag that.


In [ ]:
from tbparse import SummaryReader
import pandas as pd, glob

ckpt_dir = os.path.join(WORK, "ckpts_v2_fixed")
runs = sorted(glob.glob(os.path.join(ckpt_dir, "runs", "*")))
print(f"runs found: {len(runs)}")
for r in runs[-3:]:
    print(f"\n=== {os.path.basename(r)} ===")
    df = SummaryReader(r, pivot=True).scalars
    if df is None or df.empty:
        print("  (empty)")
        continue
    df = df.sort_values("step").reset_index(drop=True)
    cols = ["step"] + [c for c in [
        "train/loss","train/reward","train/reward_std","train/kl",
        "train/grad_norm","train/learning_rate","train/completion_length"
    ] if c in df.columns]
    print(df[cols].to_string(index=False))


## 12. (Optional) Quick eval — random vs trained, n=3 episodes each

Sanity check: does the LoRA do anything different from random? Just 3
episodes per side — enough to see directionality, not enough for a paper
claim. Full eval is in `training/eval.py`.


In [ ]:
# Skip if you're rate-limited; this is a 3-min sanity check.
RUN_QUICK_EVAL = True
if RUN_QUICK_EVAL:
    import importlib, training.eval as _eval_mod
    importlib.reload(_eval_mod)
    sys.argv = [
        "eval.py",
        "--model-name", "Qwen/Qwen2.5-3B-Instruct",
        "--adapter-path", os.path.join(WORK, "ckpts_v2_fixed"),
        "--num-baseline", "3",
        "--num-trained", "3",
        "--output-json", os.path.join(WORK, "ckpts_v2_fixed", "eval_quick.json"),
    ]
    try:
        _eval_mod.main()
    except SystemExit: pass
    except Exception as e:
        print(f"eval failed (non-fatal): {type(e).__name__}: {e}")
